In [18]:
# Import gurobi and numpy
from gurobipy import *
import numpy as np
from numpy import genfromtxt
import csv

## Get index of 4 tickers
tick4 = ["MSFT","GS","PG","SCHP"];

# Get variable names
with open('Prices.csv') as csvFile:
    reader = csv.reader(csvFile)
    tickers = next(reader) ## stores the tickers of all 390 stocks

tickind =[];
for t in tick4:
    tickind.append(tickers.index(t)) ## retrieve index that corresponds to each ticker

# Load data
prices = genfromtxt('Prices.csv', delimiter=',',skip_header = 1)

# get dimensions of data
d = prices.shape[0]
n = prices.shape[1]

# calculate monthly returns of each stock
returns = np.zeros((d-1,n))
for stock in range(n):
    for month in range(d-1):
        returns[month,stock] = prices[month+1,stock]/prices[month,stock]-1

# Store average return (parameter r_i in portfolio optimization model)
avg_return = np.zeros(n)
avg_return = np.mean(returns,axis=0)

# Store covariance matrix (parameter C_ij in portfolio optimization model)
C = np.zeros((n,n))
C = np.cov(np.transpose(returns))

In [20]:
# Preprocessed data
tickind = [tickers.index(t) for t in tick4]  # Indices for the four stocks

# Model 1: Four-asset minimum variance portfolio
C_sub = C[np.ix_(tickind, tickind)]  # Covariance submatrix for the four stocks
avg_return_sub = avg_return[tickind]  # Average returns for the four stocks

# Create Gurobi model for Model 1
m1 = Model("MinVariancePortfolio_Model1")
w = m1.addVars(len(tick4), lb=0, ub=1, name="w")  # Variables for stock weights

# Objective: Minimize portfolio variance
m1.setObjective(
    quicksum(w[i] * C_sub[i, j] * w[j] for i in range(len(tick4)) for j in range(len(tick4))),
    GRB.MINIMIZE
)

# Constraints
m1.addConstr(quicksum(w[i] for i in range(len(tick4))) == 1, "FullyInvested")
m1.addConstr(quicksum(w[i] * avg_return_sub[i] for i in range(len(tick4))) >= 0.005, "ReturnConstraint")

# Optimize the model
m1.optimize()

# Print results for Model 1
print("Model 1 Optimal Portfolio:")
for i in range(len(tick4)):
    print(f"Weight in {tick4[i]}: {w[i].X:.4f}")
print(f"Optimal risk: {m1.ObjVal:.6f}")
print(f"Solver time: {m1.Runtime:.4f} sec")

Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (mac64[x86] - Darwin 21.6.0 21G651)

CPU model: Intel(R) Core(TM) i5-8257U CPU @ 1.40GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 2 rows, 4 columns and 8 nonzeros
Model fingerprint: 0x23731cce
Model has 10 quadratic objective terms
Coefficient statistics:
  Matrix range     [2e-04, 1e+00]
  Objective range  [0e+00, 0e+00]
  QObjective range [5e-05, 7e-03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e-03, 1e+00]
Presolve time: 0.01s
Presolved: 2 rows, 4 columns, 8 nonzeros
Presolved model has 10 quadratic objective terms
Ordering time: 0.00s

Barrier statistics:
 Free vars  : 3
 AA' NZ     : 1.000e+01
 Factor NZ  : 1.500e+01
 Factor Ops : 5.500e+01 (less than 1 second per iteration)
 Threads    : 1

                  Objective                Residual
Iter       Primal          Dual         Primal    Dual     Compl     Time
   0   2.93770406e+03 -2.93770406e+03  4.00e+03 

In [22]:
# Model 2: Minimum-variance portfolio with all 390 stocks
m2 = Model("MinVariancePortfolio_Model2")
w2 = m2.addVars(n, lb=0, ub=1, name="w_all") 

# Objective: Minimize portfolio variance
m2.setObjective(
    quicksum(w2[i] * C[i, j] * w2[j] for i in range(n) for j in range(n)),
    GRB.MINIMIZE
)

# Constraints
m2.addConstr(quicksum(w2[i] for i in range(n)) == 1, "FullyInvested")
m2.addConstr(quicksum(w2[i] * avg_return[i] for i in range(n)) >= 0.005, "ReturnConstraint")

# Optimize the model
m2.optimize()

# Print results for Model 2
print("\nModel 2 Optimal Portfolio:")
print(f"Optimal risk: {m2.ObjVal:.6f}")
print(f"Solver time: {m2.Runtime:.4f} sec")


Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (mac64[x86] - Darwin 21.6.0 21G651)

CPU model: Intel(R) Core(TM) i5-8257U CPU @ 1.40GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 2 rows, 390 columns and 780 nonzeros
Model fingerprint: 0x057f3f75
Model has 76245 quadratic objective terms
Coefficient statistics:
  Matrix range     [1e-06, 1e+00]
  Objective range  [0e+00, 0e+00]
  QObjective range [2e-07, 8e-02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e-03, 1e+00]
Presolve time: 0.01s
Presolved: 2 rows, 390 columns, 780 nonzeros
Presolved model has 76245 quadratic objective terms
Ordering time: 0.00s

Barrier statistics:
 Free vars  : 59
 AA' NZ     : 1.830e+03
 Factor NZ  : 1.891e+03
 Factor Ops : 7.753e+04 (less than 1 second per iteration)
 Threads    : 4

                  Objective                Residual
Iter       Primal          Dual         Primal    Dual     Compl     Time
   0   1.10520633e-17 -1.10520633

In [25]:
# Model 3: Minimum-variance portfolio selecting at most 4 stocks
m3 = Model("MinVariancePortfolio_Model3")
w3 = m3.addVars(n, lb=0, ub=1, name="w")
x = m3.addVars(n, vtype=GRB.BINARY, name="x") 

# Objective: Minimize portfolio variance
m3.setObjective(
    quicksum(w3[i] * C[i, j] * w3[j] for i in range(n) for j in range(n)),
    GRB.MINIMIZE
)

# Constraints
m3.addConstr(quicksum(w3[i] * avg_return[i] for i in range(n)) >= 0.005, "ReturnConstraint")
m3.addConstr(quicksum(x[i] for i in range(n)) <= 4, "StockLimitto4")
m3.addConstr(quicksum(w3[i] for i in range(n)) == 1, "FullyInvested")

# wi should be smaller than xi
for i in range(n):
    m3.addConstr(w3[i] <= x[i])

# Optimize the model
m3.optimize()

# Print results for Model 3
print("\nModel 3 Optimal Portfolio:")
selected_stocks_result = []
for i in range(n):
    if x[i].X > 0.5:  # Stock is selected
        selected_stocks_result.append((tickers[i], w3[i].X))
        print(f"Weight in {tickers[i]}: {w3[i].X:.4f}")

print(f"Optimal risk (variance): {m3.ObjVal:.6f}")
print(f"Solver time: {m3.Runtime:.4f} sec")


Gurobi Optimizer version 12.0.0 build v12.0.0rc1 (mac64[x86] - Darwin 21.6.0 21G651)

CPU model: Intel(R) Core(TM) i5-8257U CPU @ 1.40GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 393 rows, 780 columns and 1950 nonzeros
Model fingerprint: 0x99742bc5
Model has 76245 quadratic objective terms
Variable types: 390 continuous, 390 integer (390 binary)
Coefficient statistics:
  Matrix range     [1e-06, 1e+00]
  Objective range  [0e+00, 0e+00]
  QObjective range [2e-07, 8e-02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e-03, 4e+00]
Found heuristic solution: objective 0.0136011
Presolve time: 0.03s
Presolved: 393 rows, 780 columns, 1950 nonzeros
Presolved model has 76245 quadratic objective terms
Variable types: 390 continuous, 390 integer (390 binary)

Root relaxation: objective 2.878501e-05, 129 iterations, 0.01 seconds (0.01 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl 